# Baselines formales y regresión regularizada — 6 meses (enero-junio 2025)


Este notebook corre los baselines sobre los 6 meses completos usando el pipeline construido en la semana 8 (src/data.py, src/features.py) y agrega regresión lineal regularizada (Ridge y Lasso) como exige esta semana.



In [1]:
import sys
sys.path.append('..') 

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_log_error

from src.data import cargar_todo, split_temporal, MESES
from src.features import construir_preprocesador, preparar_xy

SEED = 42
np.random.seed(SEED)

import src.features as features
features.RUTA_ZONAS = Path('../data/Taxi_Zone_Lookup.csv')
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

## 1. Carga de los 6 meses
Con sample_frac=0.20 nos quedamos con ~20% de cada mes (~3.6M filas en total).

In [2]:
SAMPLE_FRAC = 0.20

df = cargar_todo(meses=MESES, sample_frac=SAMPLE_FRAC, seed=SEED)
print(f"Filas tras limpieza y muestreo ({SAMPLE_FRAC:.0%} por mes): {len(df):,}")
df['mes'].value_counts().sort_index()

Filas tras limpieza y muestreo (20% por mes): 4,531,075


mes
2025-01    659521
2025-02    678912
2025-03    784902
2025-04    749106
2025-05    855412
2025-06    803222
Name: count, dtype: int64

## 2. Split temporal

In [3]:
train, val, test = split_temporal(df)
print(f"Train (ene-abr): {len(train):,}")
print(f"Val   (mayo):    {len(val):,}")
print(f"Test  (junio):   {len(test):,}")

y_tr = train['trip_duration_min']
y_val = val['trip_duration_min']

Train (ene-abr): 2,872,441
Val   (mayo):    855,412
Test  (junio):   803,222


## 3. Función de evaluación
MAE como métrica principal, RMSLE como secundaria

In [4]:
def evaluar(nombre, y_real, y_pred, resultados):
    y_pred = np.clip(y_pred, 0, None)
    mae = mean_absolute_error(y_real, y_pred)
    rmsle = np.sqrt(mean_squared_log_error(y_real, y_pred))
    print(f"{nombre:<42} MAE = {mae:6.2f} min | RMSLE = {rmsle:.4f}")
    resultados.append({'modelo': nombre, 'MAE': mae, 'RMSLE': rmsle})
    return resultados

resultados = []

## 4. Baselines simples (niveles A y B)
Replicamos los del notebook 01_exploracion_inicial.ipynb, ahora sobre los 6 meses.

In [5]:
# A. Mediana global
pred_a = np.full(len(val), y_tr.median())
resultados = evaluar("A: Mediana global", y_val, pred_a, resultados)

A: Mediana global                          MAE =   9.28 min | RMSLE = 0.6650


In [6]:
# B. Mediana por (zona de recojo, hora)
train = train.copy()
val = val.copy()
train['hora'] = train['tpep_pickup_datetime'].dt.hour
val['hora'] = val['tpep_pickup_datetime'].dt.hour

tabla = train.groupby(['PULocationID', 'hora'])['trip_duration_min'].median()
pred_b = val.set_index(['PULocationID', 'hora']).index.map(tabla).to_numpy(dtype=float)
pred_b = np.where(np.isnan(pred_b), y_tr.median(), pred_b)
resultados = evaluar("B: Mediana por (zona, hora)", y_val, pred_b, resultados)

B: Mediana por (zona, hora)                MAE =   7.86 min | RMSLE = 0.5804


## 5. Regresión lineal simple vs. regularizada 

construir_preprocesador() (de src/features.py) aplica ExtractorFeaturesNYC (features temporales, geográficas y distancia estimada sin leakage) más la codificación (TargetEncoder para zonas, OneHot para categóricas de baja cardinalidad, StandardScaler para numéricas). Se ajusta solo con train, como exige el control de leakage del proposal.

In [7]:
X_tr, y_tr_arr = preparar_xy(train, target_col='trip_duration_min')
X_val, y_val_arr = preparar_xy(val, target_col='trip_duration_min')

modelos = [
    ("C: Regresión lineal (sin regularizar)", LinearRegression()),
    ("D: Ridge (regularizada, alpha=1.0)", Ridge(alpha=1.0, random_state=SEED)),
    ("E: Lasso (regularizada, alpha=0.01)", Lasso(alpha=0.01, random_state=SEED)),
]

pipelines_ajustados = {}

for nombre, modelo in modelos:
    pipe = Pipeline([
        ('features', construir_preprocesador(escalar_numericas=True)),
        ('modelo', modelo),
    ])
    pipe.fit(X_tr, y_tr_arr)
    pred = pipe.predict(X_val)
    resultados = evaluar(nombre, y_val_arr, pred, resultados)
    pipelines_ajustados[nombre] = pipe

C: Regresión lineal (sin regularizar)      MAE =   5.14 min | RMSLE = 0.3887
D: Ridge (regularizada, alpha=1.0)         MAE =   5.14 min | RMSLE = 0.3887
E: Lasso (regularizada, alpha=0.01)        MAE =   5.14 min | RMSLE = 0.3870


## 6. Barrido simple de alpha para Ridge


In [8]:
alphas = [0.1, 1.0, 10.0, 50.0]

for alpha in alphas:
    pipe = Pipeline([
        ('features', construir_preprocesador(escalar_numericas=True)),
        ('modelo', Ridge(alpha=alpha, random_state=SEED)),
    ])
    pipe.fit(X_tr, y_tr_arr)
    pred = pipe.predict(X_val)
    resultados = evaluar(f"Ridge alpha={alpha}", y_val_arr, pred, resultados)

Ridge alpha=0.1                            MAE =   5.14 min | RMSLE = 0.3887
Ridge alpha=1.0                            MAE =   5.14 min | RMSLE = 0.3887
Ridge alpha=10.0                           MAE =   5.14 min | RMSLE = 0.3886
Ridge alpha=50.0                           MAE =   5.14 min | RMSLE = 0.3886


## 7. Tabla final y guardado de métricas

In [9]:
tabla_resultados = pd.DataFrame(resultados).sort_values('MAE').reset_index(drop=True)
tabla_resultados

,modelo,MAE,RMSLE
0,Ridge alpha=10.0,5.143932,0.388641
1,"D: Ridge (regularizada, alpha=1.0)",5.143940,0.388650
2,Ridge alpha=1.0,5.143940,0.388650
3,Ridge alpha=0.1,5.143941,0.388651
4,C: Regresión lineal (sin regularizar),5.143942,0.388651
5,Ridge alpha=50.0,5.143946,0.388603
6,"E: Lasso (regularizada, alpha=0.01)",5.144750,0.387012
7,"B: Mediana por (zona, hora)",7.858684,0.580354
8,A: Mediana global,9.283539,0.664978


In [10]:
Path('../outputs').mkdir(exist_ok=True)
with open('../outputs/metrics_6meses.json', 'w') as f:
    json.dump(resultados, f, indent=2)
print("Guardado en ../outputs/metrics_6meses.json")

Guardado en ../outputs/metrics_6meses.json


## 8. Lectura de resultados y comparación con enero

En el notebook 01_exploracion_inicial.ipynb (solo enero, sin las features de src/features.py), el mejor baseline fue la regresión lineal simple con MAE = 6.08 min.

Con los 6 meses y el pipeline completo (src/data.py + src/features.py), los resultados sobre el conjunto de validación (mayo) fueron:

| Modelo                        | MAE (min) | RMSLE  |
|--------------------------------|-----------|--------|
| A. Mediana global               | 9.28      | 0.6650 |
| B. Mediana por (zona, hora)     | 7.86      | 0.5804 |
| C. Regresión lineal             | 5.14      | 0.3887 |
| D. Ridge (alpha 0.1 a 50)       | 5.14      | 0.3886-0.3887 |
| E. Lasso (alpha=0.01)           | 5.14      | 0.3870 |

El salto más grande está entre los baselines simples y los modelos lineales con features: de 9.28 min (mediana global) a 5.14 min, una mejora del 45%. Esto confirma que las features construidas en src/features.py capturan señal real que los baselines por grupos simples (B) no alcanzan a explotar del todo.

La regularización (Ridge y Lasso) prácticamente no cambia el resultado frente a la regresión lineal simple: las diferencias de MAE entre C, D y E están en el tercer o cuarto decimal, y el barrido de alpha (0.1 a 50) para Ridge tampoco mueve la aguja. Es un resultado esperable: con ~2.9M de filas de entrenamiento, el modelo lineal no tiene margen real para sobreajustar, así que penalizar los coeficientes aporta poco. Esto no invalida la regularización como práctica correcta.

Nota sobre reproducibilidad: SAMPLE_FRAC=0.20, SEED=42 y el split temporal (train ene-abr, val mayo, test junio) quedan fijos y documentados en las celdas anteriores. Los resultados completos están en outputs/metrics_6meses.json.

